In [0]:
from pyspark.sql import Row

#simulating an incoming update batch - 3 employees got transferred/changed
updates = spark.createDataFrame([
    Row(employee_id=1, Department="Research & Development", JobRole="Research Scientist"),
    Row(employee_id=2, Department="Human Resources", JobRole="HR Coordinator"),
    Row(employee_id=4, Department="Sales", JobRole="Sales Executive"),
])

updates.createOrReplaceTempView("incoming_updates")

In [0]:
%sql
--now apply SCD Type 1 using a MERGE - this is the real-world syntax for an overwrite/UPSERT:

MERGE INTO ibm_hr.silver.dim_employee AS target
USING incoming_updates AS source
ON target.employee_id = source.employee_id
WHEN MATCHED THEN
  UPDATE SET
    target.Department = source.Department,
    target.JobRole = source.JobRole

In [0]:
#verifying the change landed and the old value is gone
spark.sql("SELECT * FROM ibm_hr.silver.dim_employee WHERE employee_id IN (1,2,4)").show()